# Scenario 10: OpenTelemetry + OpenInference Integration with Galileo

This notebook shows the **supported Galileo OTel integration path** for teams that already use OpenTelemetry and want Galileo as an OTLP backend for AI traces.

By the end, you'll understand how to:

1. **Create a standard OTel `TracerProvider`** and attach Galileo's `GalileoSpanProcessor`
2. **Auto-instrument OpenAI with OpenInference** for AI-aware spans and attributes
3. **Send LLM traces to Galileo without using the `galileo.openai` wrapper**
4. **Create custom workflow, retriever, and tool spans** that nest around auto-instrumented LLM calls
5. **Refactor repeated span logic into reusable decorators**
6. **Flush and shut down the pipeline cleanly** in notebook-style code

## Prerequisites

- A `.env` file with `GALILEO_API_KEY` and `OPENAI_API_KEY`
- Dependencies installed via `uv sync`


## Step 0: Load Environment Variables


In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')


Loaded credentials from /Users/binliu/Documents/rungalileo/galileo-test/.env


## Step 0b: Imports and Constants

The tracing path now follows the Galileo docs and the repo's Scenario 10 plan:

- `galileo.otel.GalileoSpanProcessor` handles the Galileo OTLP exporter setup
- `OpenAIInstrumentor` from **OpenInference** captures OpenAI spans automatically
- `start_galileo_span` maps Galileo workflow, retriever, and tool span types onto OTel spans

### How Galileo extracts fields from OTel spans

Galileo's OTLP backend automatically maps span attributes (no per-project configuration):

| Field | Attribute (primary) | Fallback |
|---|---|---|
| Input/output messages | `gen_ai.input.messages` / `gen_ai.output.messages` | Span events (`gen_ai.user.message`, `gen_ai.assistant.message`) |
| System instructions | `gen_ai.system_instructions` | First system message in `gen_ai.input.messages` |
| Token metrics | `gen_ai.usage.input_tokens` / `gen_ai.usage.output_tokens` | OpenInference `llm.token_count.prompt` / `.completion` |
| Context/documents | Retriever span output (list of documents) | Extracted automatically from retriever spans |
| Ground truth | `galileo.dataset.input` / `galileo.dataset.output` | Resource attributes for evaluation metrics |


In [2]:
import functools
import os

import openai
from galileo import galileo_context, otel
from galileo.config import GalileoPythonConfig
from galileo.projects import delete_project
from galileo_core.schemas.logging.span import RetrieverSpan, ToolSpan, WorkflowSpan
from galileo_core.schemas.shared.document import Document
from openinference.instrumentation.openai import OpenAIInstrumentor
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider

PROJECT_NAME = 'GalileoEval_S10_OTel'
LOG_STREAM_NAME = 'otel-integration'
SERVICE_NAME = 'galileo-otel-demo'
MODEL = 'gpt-4o-mini'

PROJECT_NAME, LOG_STREAM_NAME, SERVICE_NAME, MODEL


('GalileoEval_S10_OTel',
 'otel-integration',
 'galileo-otel-demo',
 'gpt-4o-mini')

## Step 1: Initialize Galileo

`galileo_context.init()` still bootstraps the project and log stream. After that we mirror the resolved values into environment variables so the OTel path and the Galileo SDK agree about where traces should land.


In [3]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

os.environ['GALILEO_PROJECT'] = PROJECT_NAME
os.environ['GALILEO_LOG_STREAM'] = LOG_STREAM_NAME
os.environ['OTEL_SERVICE_NAME'] = SERVICE_NAME

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    log_stream_url = 'Galileo context initialized, but no log stream URL was returned.'
    print(log_stream_url)


https://console.demo-v2.galileocloud.io/project/f2d4f0ec-0141-45e6-ae7e-8111e9926ff8
https://console.demo-v2.galileocloud.io/project/f2d4f0ec-0141-45e6-ae7e-8111e9926ff8/log-streams/5bff557a-244d-4d87-a33b-f6473a68e6c9


## Step 2: Configure the OTel Pipeline

The key simplification in this refactor is that we no longer build an `OTLPSpanExporter` by hand or maintain a custom content bridge.

Instead:

1. `TracerProvider` is still standard OTel
2. `GalileoSpanProcessor` wires the exporter to Galileo and injects routing metadata
3. `OpenAIInstrumentor` from OpenInference creates AI-aware spans for OpenAI calls

The helper below also makes notebook re-runs less fragile by uninstrumenting before re-instrumenting.


In [4]:
def configure_tracing() -> tuple[TracerProvider, otel.GalileoSpanProcessor, OpenAIInstrumentor]:
    resource = Resource.create(
        {
            'service.name': SERVICE_NAME,
            'service.version': '1.0.0',
            'deployment.environment': 'notebook',
        }
    )

    tracer_provider = TracerProvider(resource=resource)
    span_processor = otel.GalileoSpanProcessor(
        project=PROJECT_NAME,
        logstream=LOG_STREAM_NAME,
    )
    otel.add_galileo_span_processor(tracer_provider, span_processor)

    instrumentor = OpenAIInstrumentor()
    try:
        instrumentor.uninstrument()
    except Exception:
        pass
    instrumentor.instrument(tracer_provider=tracer_provider)

    return tracer_provider, span_processor, instrumentor


def force_flush(timeout_millis: int = 40000) -> None:
    span_processor.force_flush(timeout_millis)


tracer_provider, span_processor, instrumentor = configure_tracing()
client = openai.OpenAI()

print('OTel pipeline ready:')
print(f'  Project      -> {PROJECT_NAME}')
print(f'  Log stream   -> {LOG_STREAM_NAME}')
print('  Export path  -> GalileoSpanProcessor-managed OTLP exporter')
print('  Instrumentor -> OpenInference OpenAIInstrumentor')


Attempting to uninstrument while already uninstrumented


OTel pipeline ready:
  Project      -> GalileoEval_S10_OTel
  Log stream   -> otel-integration
  Export path  -> GalileoSpanProcessor-managed OTLP exporter
  Instrumentor -> OpenInference OpenAIInstrumentor


## Step 3: Make an LLM Call

This first call exercises the simplest path: OpenAI is auto-instrumented by OpenInference, and Galileo receives the resulting OTel trace through the Galileo span processor.


In [5]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {'role': 'system', 'content': 'You are a helpful assistant that explains technical concepts clearly.'},
        {'role': 'user', 'content': 'What is OpenTelemetry and why is it useful for AI applications?'},
    ],
    max_tokens=200,
)

force_flush()

print('Response:')
print(response.choices[0].message.content)
print(f'\nTokens: {response.usage.prompt_tokens} in / {response.usage.completion_tokens} out')
print(f'\n-> Trace flushed to Galileo. Check: {log_stream_url}')


Response:
OpenTelemetry is an open-source observability framework designed to help developers collect, process, and export telemetry data from their applications. It provides standardized APIs and libraries that facilitate the gathering of three key types of telemetry data: metrics (quantitative measurements), traces (distributed transaction data), and logs (event and error messages). Essentially, OpenTelemetry enables better observability of applications, which is crucial for monitoring performance, debugging issues, and improving user experience.

### Key Features of OpenTelemetry:

1. **Interoperability**: OpenTelemetry is designed to work with various programming languages and frameworks, making it easier to instrument applications no matter the stack.

2. **Standardization**: It offers a unified way to collect telemetry data. This means developers can use consistent formats and protocols across different services, which simplifies integration with various observability tools and p

## Step 4: Custom Workflow, Retriever, and Tool Spans

The OpenAI call in this step is the **LLM span**. Because `OpenAIInstrumentor` is already attached to the shared `TracerProvider`, `client.chat.completions.create(...)` is automatically logged as a child LLM span inside the surrounding workflow trace.

We add custom workflow, retriever, and tool spans around it with `otel.start_galileo_span(...)`, giving one nested trace with four span types:

- workflow span
- retriever span
- LLM span
- tool span


In [6]:
user_question = 'What are the best practices for OpenTelemetry in LLM apps?'
workflow = WorkflowSpan(name='research-agent', input=user_question)

with otel.start_galileo_span(workflow):
    retrieved = [
        Document(content='Always use a BatchSpanProcessor in production.', metadata={'source': 'otel-docs', 'score': 0.91}),
        Document(content='Set OTEL_SERVICE_NAME so spans carry a service identity.', metadata={'source': 'otel-docs', 'score': 0.87}),
        Document(content='Use a vendor-provided span processor when the backend needs routing metadata.', metadata={'source': 'galileo-docs', 'score': 0.84}),
    ]
    retriever = RetrieverSpan(name='vector-search', input=user_question, output=retrieved)
    with otel.start_galileo_span(retriever):
        print(f'Retrieved {len(retrieved)} documents')

    # This OpenAI call is auto-instrumented as the trace's LLM span.
    context_text = '\n'.join(f'- {doc.content}' for doc in retrieved)
    synthesis = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Answer concisely based on the provided context.'},
            {'role': 'user', 'content': f'Context:\n{context_text}\n\nQuestion: {user_question}'},
        ],
        max_tokens=150,
    )
    answer = synthesis.choices[0].message.content

    tool = ToolSpan(name='format-final-answer', input=answer, tool_call_id='call_format_001')
    with otel.start_galileo_span(tool):
        formatted = answer.strip()
        tool.output = formatted

    workflow.output = formatted

force_flush()

print(f'\nFinal answer: {formatted}')
print('\n-> One trace exported with workflow -> retriever -> llm -> tool spans')
print(f'   Check: {log_stream_url}')


Retrieved 3 documents

Final answer: Best practices for OpenTelemetry in LLM apps include:

1. Use a **BatchSpanProcessor** in production for efficient span processing.
2. Set the **OTEL_SERVICE_NAME** environment variable to ensure spans carry a service identity.
3. Utilize a **vendor-provided span processor** when routing metadata to the backend is necessary.

-> One trace exported with workflow -> retriever -> llm -> tool spans
   Check: https://console.demo-v2.galileocloud.io/project/f2d4f0ec-0141-45e6-ae7e-8111e9926ff8/log-streams/5bff557a-244d-4d87-a33b-f6473a68e6c9


## Step 5: Reusable Decorators

Once the pipeline is working, the next cleanup is to stop repeating span boilerplate. These decorators keep the Galileo span types explicit while making notebook and application code much easier to read.


In [27]:
def _as_documents(values):
    documents = []
    for value in values:
        if isinstance(value, Document):
            documents.append(value)
        else:
            documents.append(Document(content=str(value)))
    return documents


def otel_tool(name):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            span_data = ToolSpan(name=name, input=f'args={args} kwargs={kwargs}')
            with otel.start_galileo_span(span_data):
                result = fn(*args, **kwargs)
                span_data.output = str(result)
                return result

        return wrapper

    return decorator


def otel_retriever(name):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            query = args[0] if args else kwargs.get('query', '')
            span_data = RetrieverSpan(name=name, input=str(query), output=[])
            with otel.start_galileo_span(span_data):
                result = fn(*args, **kwargs)
                span_data.output = _as_documents(result)
                return result

        return wrapper

    return decorator


def otel_span(name):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            span_data = WorkflowSpan(name=name, input=f'args={args} kwargs={kwargs}')
            with otel.start_galileo_span(span_data):
                result = fn(*args, **kwargs)
                span_data.output = str(result)
                return result

        return wrapper

    return decorator


print('Decorators defined: @otel_tool, @otel_retriever, @otel_span')


Decorators defined: @otel_tool, @otel_retriever, @otel_span


## Step 6: Use the Decorators in a Workflow

This final example keeps the OpenAI call auto-instrumented while the surrounding steps use the reusable custom span decorators.


In [28]:
@otel_retriever('search-knowledge-base')
def search_knowledge_base(query):
    return [
        'TracerProvider owns span processors and tracers.',
        'GalileoSpanProcessor routes spans to the correct Galileo project and log stream.',
        'OpenInference adds AI-specific tracing for OpenAI calls without using galileo.openai.',
    ]


@otel_tool('format-context')
def format_context(docs):
    return '\n'.join(f'- {doc}' for doc in docs)


@otel_tool('format-final-output')
def format_final_output(answer):
    return answer.strip()


@otel_span('knowledge-qa-pipeline')
def answer_question(query):
    docs = search_knowledge_base(query)
    context = format_context(docs)

    llm_response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': 'Answer concisely based on the context provided.'},
            {'role': 'user', 'content': f'Context:\n{context}\n\nQuestion: {query}'},
        ],
        max_tokens=100,
    )

    return format_final_output(llm_response.choices[0].message.content)


final = answer_question('What is a TracerProvider?')
force_flush()

print(f'Answer: {final}')
print('\n-> Pipeline trace flushed to Galileo')
print(f'   Check: {log_stream_url}')


Answer: A TracerProvider is a component that manages span processors and tracers, facilitating the creation and handling of traces within a tracing system.

-> Pipeline trace flushed to Galileo
   Check: https://console.demo-v2.galileocloud.io/project/f2d4f0ec-0141-45e6-ae7e-8111e9926ff8/log-streams/5bff557a-244d-4d87-a33b-f6473a68e6c9


## Step 7: Flush and Verify

`BatchSpanProcessor` exports asynchronously. In notebook code, an explicit flush makes the traces show up in Galileo before you move on.


In [ ]:
force_flush()

print('All spans flushed to Galileo')
print(f'View traces and metrics at: {log_stream_url}')


## Step 8: Clean Up

Clean shutdown matters for OTel as much as setup:

1. Shut down the `TracerProvider` to drain the batch processor
2. Uninstrument OpenAI so later notebook cells or reruns start cleanly
3. Delete the Galileo project when you're done exploring


In [ ]:
tracer_provider.shutdown()
instrumentor.uninstrument()

delete_project(name=PROJECT_NAME)
print(f'Deleted project: {PROJECT_NAME}')
